# Tutorial 4: Posterior Analysis and Calibration

After running MCMC we need to:
1. Load and flatten the chain (removing burn-in)
2. Make a **corner plot** showing all pairwise marginals
3. Extract point estimates (MAP, median) and credible intervals
4. Run a **closure test** to verify the posterior is well-calibrated

The calibration metrics (Mahalanobis distance, Energy Score, KNN log-density)
come from Gabriel_TLProject and allow quantitative comparison of analysis pipelines.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hic_bayes import (
    load_chain_hdf5, thin_chain,
    map_estimate, credible_interval, posterior_summary, corner_plot,
    calibration_report,
)
from hic_bayes.plots import plot_calibration_report

param_names  = ['eta_s', 'zeta_s_max', 'tau_0', 'Qs_mu', 'T_switch']
param_labels = [
    r'$\eta/s$', r'$(\zeta/s)_{\rm max}$',
    r'$\tau_0$', r'$Q_s/\mu$', r'$T_{\rm switch}$',
]

# Load the chain saved in Tutorial 3
try:
    chain, log_prob_arr = load_chain_hdf5('../results/chain_tutorial.hdf5')
    print(f'Chain loaded: {chain.shape}')
except FileNotFoundError:
    print('Run Tutorial 3 first to generate the chain file.')

In [ ]:
BURNIN = chain.shape[1] // 2
THIN   = 2

flat_chain    = thin_chain(chain, burnin=BURNIN, thin=THIN)
flat_log_prob = log_prob_arr[:, BURNIN::THIN].ravel()

print(f'Flat chain: {flat_chain.shape[0]} samples')

In [ ]:
summary = posterior_summary(flat_chain, param_names, level=0.90)

print(f'\n{"Parameter":<14} {"Mean":>8}  {"Std":>7}  {"90% CI":>20}')
print('-' * 55)
for name, s in summary.items():
    ci = f"[{s['ci_low']:.4f}, {s['ci_high']:.4f}]"
    print(f"{name:<14} {s['mean']:>8.4f}  {s['std']:>7.4f}  {ci:>20}")

theta_map = map_estimate(flat_chain, flat_log_prob)
print(f'\nMAP: {dict(zip(param_names, theta_map.round(4)))}')

In [ ]:
# Load true parameters for the closure test
X_val   = pd.read_csv('../data/toy/validation_design.csv').values
true_params = X_val[10]   # index matching config closure_test.design_index

fig = corner_plot(flat_chain, param_labels, truths=true_params, title='Toy model posterior')
plt.show()

In [ ]:
report = calibration_report(true_params, flat_chain, param_names)

print('\n=== Calibration report ===')
print(f'  Mahalanobis distance : {report["mahalanobis"]:.3f}')
print(f'  (well-calibrated ~ {len(param_names)} for chi2({len(param_names)}) distribution)')
print(f'  Energy Score         : {report["energy_score"]:.4f}')
print(f'  KNN log-density      : {report["log_density"]:.3f}')

fig = plot_calibration_report(report)
plt.show()